In [ ]:
import os
import pty
import fcntl
import termios
import struct
import select
import codecs
import subprocess
import sys

def run(command):
    master_descriptor, slave_descriptor = pty.openpty()

    try:
      terminal_columns, terminal_rows = os.get_terminal_size()
    except OSError:
      terminal_columns, terminal_rows = 120, 30

    try:
      fcntl.ioctl(slave_descriptor, termios.TIOCSWINSZ,
                  struct.pack("HHHH", terminal_rows, terminal_columns, 0, 0))
    except OSError:
      pass

    child_process = subprocess.Popen(
        command,
        shell=isinstance(command, str),
        stdin=slave_descriptor,
        stdout=slave_descriptor,
        stderr=slave_descriptor,
        close_fds=True,
        preexec_fn=os.setsid,
    )
    os.close(slave_descriptor)

    incremental_utf8_decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")

    while True:
      readable, _, _ = select.select([master_descriptor], [], [], 0.2)

      if readable:
        try:
          raw_bytes = os.read(master_descriptor, 4096)
        except OSError:
          break

        if not raw_bytes:
          break

        sys.stdout.write(incremental_utf8_decoder.decode(raw_bytes))
        sys.stdout.flush()
      elif child_process.poll() is not None:
        break

    os.close(master_descriptor)
    child_process.wait()

    if child_process.returncode != 0:
      raise subprocess.CalledProcessError(child_process.returncode, command)
    
    return child_process.returncode

# Set this to wherever the python binary is located
PYTHON_BIN = ".venv/bin/python"

# Experiment 001, Typo Robustness: Colab driver (dress rehearsal)

Full pipeline in seven cells:
**clone → install → auth → build items → build dictionary → generate (full roster) → analyze + download**

The pilot (v2) is complete and its outputs are committed; this driver now runs
the **full-run dress rehearsal** (`configs/rehearsal.yaml`, design/00 §0.5
2026-07-21): every roster model through the complete main-study condition grid
at 24 items per dataset, then one combined analysis. Scoring is performed
inline during generation (no separate scoring step).

**Division of labor:** Colab (T4) is for testing the machinery; the USC GPU
cluster runs whatever the T4 cannot. The fp16 7-8B models (`mistral_7b`,
`qwen_7b`, `llama_8b`) do not fit a T4's 16 GB; Cell 5 records them as failed
and moves on. The same output directory, moved between machines, resumes
seamlessly (see below).

**Before starting:** Runtime → Change runtime type → T4 GPU (or better).
Enable **Notebook access** on the `HF_TOKEN` secret (key icon, left sidebar);
the token's HuggingFace account needs accepted access for
`meta-llama/Llama-3.2-1B-Instruct`, `Llama-3.2-3B-Instruct`,
`Llama-3.1-8B-Instruct`, and `mistralai/Mistral-7B-Instruct-v0.3`
(each is a separate click-through on its HF page; Qwen and the
hugging-quants AWQ repos are ungated).

**Run cells top to bottom, in order.** Environment state (the clone, the
`run` helper from Cell 1) lives on the VM and must be redone after a runtime
restart; the *run outputs* do not, because Cell 5 writes them to Google Drive.

---
### Pipeline overview

| Cell | Tool | Time (approx) | Output |
|------|------|---------------|--------|
| 1 | clone + CPU deps | 2 min | environment ready |
| 2 | GPU stack + spaCy transformer + R/lme4 | 5 min | CUDA verified |
| 2b | HF auth from Colab secret | seconds | gated repos reachable |
| 3 | `build_task_items` + `build_annotated_dataset` | 5–10 min | `data/items/*.jsonl` |
| 4 | `build_dictionary` (SCOWL) | 1 min | `data/wordlists/en_us_pinned.txt` |
| 5 | `run_generation`, all 8 roster models | hours (T4 covers 5 of 8) | `<Drive>/rehearsal/<model>/rehearsal_generations.jsonl` |
| 6 | combined `run_analysis` + `build_report` + download | ~10 min | `rehearsal_results.zip` |

Every cell streams its subprocess's stdout and stderr live as it runs.

---
### Resuming: portable across sessions and machines

Generation is idempotent and flushes every row to disk the instant it is
generated. Resume state is **only** the output directory (rows are skipped by
`row_id` already present in the JSONL; the shard manifest is a file next to
it). Nothing is tied to a VM or GPU session:

- **Same Colab account:** Cell 5 writes to Google Drive, so a recycled VM
  resumes automatically after re-running Cells 1–4.
- **Different machine (e.g. the USC cluster):** copy the model's output
  directory over and run `tools/run_generation.py` with
  `--output-directory` pointing at it; finished rows are skipped, pending
  rows are generated. The cluster can also shard one model across GPUs with
  `--shard-index`/`--shard-count` (zero coordination; workers write
  disjoint `_wXofY_` files).
- **Do not mix sharded and unsharded runs of one model in one directory.**
  Each worker skips only rows in its *own* file, so sharding a model that
  already has a partial unsharded file duplicates the finished rows. Shard
  from an empty directory, or resume unsharded.
- **Caveat (non-confirmatory runs only):** `row_id` includes the model
  revision, and the rehearsal stamps the *live* HuggingFace SHA. If a model
  repo publishes a new revision between sessions, that model regenerates
  from zero and its file then mixes rows from both revisions; delete that
  model's directory first if this happens. Pinned confirmatory runs cannot
  hit this.
- Worst case on an abrupt VM death, the last few unflushed rows are lost
  and simply regenerate on resume.

---
### Key outputs to review after Cell 6

- **`gates.json`** first: the mechanized gate readout over the combined run
- **Multi-model crossed GLMM + quantization interaction** in `analysis/rehearsal/`
- **Coverage report** in each model's `rehearsal_exclusions.jsonl`
- **`report.html`**: filterable cells, severity chart, mediation forest plot, per-item diffs

### Cell 1 — Clone repo and install CPU dependencies

In [ ]:
run("git clone -b pilot-v2 https://github.com/natSegOS/glamor-research-onboarding.git")

%cd glamor-research-onboarding/experiments/001_typo_robustness

# Run only when using normal venv conflicts with required dependencies
# run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
# run(["virtualenv", "-q", "-p", sys.executable, ".venv"])

run([PYTHON_BIN, "-u", "-m", "pip", "install", "--upgrade", "pip"])
run([PYTHON_BIN, "-u", "-m", "pip", "install", "-e", ".", "-q"])
run([PYTHON_BIN, "-u", "-m", "pip", "install", "-r", "requirements.txt", "-q"])

print("CPU deps installed")

### Cell 2 — Install GPU stack and verify CUDA

Installs the GPU generation stack from `requirements-gpu.txt` (the vLLM stack;
`transformers` is held in 4.x there), then downloads the `en_core_web_trf`
spaCy transformer model — spaCy itself already arrived via `requirements.txt`
in Cell 1.

`spacy-transformers` uses PyTorch and runs GPU-accelerated on the T4 during
the annotation step (Cell 3). The `en_core_web_trf` model is the RoBERTa-based
spaCy pipeline used for both pilot and confirmatory annotation runs — the
pilot must be methodologically identical to the main study (design/11 §11.2).

Finally, installs the confirmatory-GLMM R bridge from `requirements-stats.txt`
(`rpy2`). R ships in the Colab image; `lme4` is installed into it once per
runtime.

In [ ]:
import subprocess, sys

run([
    PYTHON_BIN, "-u", "-m", "pip", "install",
    "-r", "requirements-gpu.txt",
    "-q",
])

# Download the spaCy transformer model (GPU-accelerated via spacy-transformers).
# en_core_web_trf uses RoBERTa-base; published benchmarks at spacy.io/models/en.
# The exact model SHA is recorded in data/items/annotation_PROVENANCE.json.
run([PYTHON_BIN, "-u", "-m", "spacy", "download", "en_core_web_trf", "-q"])

run([PYTHON_BIN, "-u", "-m", "pip", "install", "-r", "requirements-stats.txt", "-q"])
run(["R", "-e", "if (!requireNamespace('lme4', quietly=TRUE)) install.packages('lme4', repos='https://cloud.r-project.org')"])

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU found — check Runtime > Change runtime type > T4 GPU")

### Cell 2b — HuggingFace authentication (gated Llama access)

`llama_1b` is a gated repo. Reads the token from the Colab secret
`HF_TOKEN` (preferred: survives runtime restarts, never printed); falls
back to an interactive `notebook_login()` prompt if the secret is absent.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token loaded from Colab secret HF_TOKEN")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()


### Cell 3 — Fetch task items and annotate with K_P(x) key terms

**`build_task_items`** downloads 100 items per dataset from HuggingFace and writes
pinned JSONL files to `data/items/`.  The `p1` GSM-Symbolic variant is used because
it is derived from 100 annotated templates in Apple's companion GitHub repo
(github.com/apple/ml-gsm-symbolic), which this cell also clones.

The Apple template repo provides the symbolic **structure** (parameter names, types,
answer formula); the HF question text provides the **instance values**.  The tool
matches the template's format string against each HF question to extract the real
parameter values, then validates `answer_function(**extracted) == gold_answer`.
Items that pass are fully Regime C capable; items that fail are excluded gracefully
and appear in `exclusions.jsonl`.  Check the `instance params validated` line in
the cell output to see the actual coverage rate.

**`build_annotated_dataset`** annotates each item with its frozen K_P(x) key-term
set using `en_core_web_trf` (design/04 §4.6).  GPU-accelerated on the T4.

In [ ]:
import subprocess, sys

# Clone Apple's GSM-Symbolic template repo.
# The templates/ and generated_data/ directories are needed to enrich the
# HF items with question_annotated fields for Regime C operand-swap.
# License: Apple custom open-source license (source/binary redistribution
# permitted); generated_data/ is CC-BY-NC-ND-4.0 (used locally, not committed).
run([
    "git", "clone", "--depth", "1",
    "https://github.com/apple/ml-gsm-symbolic.git",
    "/tmp/ml-gsm-symbolic",
])

# Fetch 100 items per dataset from HuggingFace, enrich GSM-Symbolic items
# with question_annotated and validated instance parameters from the Apple
# template repo.
run([
    PYTHON_BIN, "-u", "tools/build_task_items.py",
    "--reasoning-items",   "100",
    "--mcq-items",         "100",
    "--gsm-config",        "p1",
    "--seed",              "1729",
    "--output-directory",  "data/items",
    "--gsm-templates-dir", "/tmp/ml-gsm-symbolic",
])

# Annotate with frozen K_P(x) key terms using en_core_web_trf.
# GPU-accelerated via spacy-transformers; takes ~5 min on a T4.
run([
    PYTHON_BIN, "-u", "tools/build_annotated_dataset.py",
    "--model-name", "en_core_web_trf",
    "--items-dir",  "data/items",
    "--force",
])

print("Items annotated and ready in data/items/")

### Cell 4 — Build the SCOWL English dictionary

Downloads SCOWL 2020.12.07 (Kevin Atkinson, wordlist.aspell.net) from its
SourceForge release archive and builds `data/wordlists/en_us_pinned.txt`.

The vocabulary is restricted to the **`words`** sub-category only (english +
american dialect, size band ≤60) — not the full bundle SCOWL's own `mk-list`
tool would pull in by default (`abbreviations`, `upper`, `proper-names`,
`contractions`, and the `special` category of hacker jargon and roman
numerals). Those extra categories exist so a *spell-checker* doesn't flag
"Mr.", "TCP", or "IV" as misspelled — a different question from what
`is_word` needs to answer (is this edited token a real word a reader would
recognize as distinct and meaningful — the check that separates Regime A
from Regime B). Measured effect of the full bundle: it counts 100% of single
letters, 51.8% of two-letter strings, and 7.3% of three-letter strings as
"real words" (mostly lower-cased abbreviations and roman numerals), which
inflates false "landed on a real word" rejections when constructing Regime A
items. The `words`-only list drops that to 11.2% / 3.8% for two/three-letter
strings while remaining exactly as citable and reproducible (still the sole
source, still one dialect, still SHA-pinned in `PROVENANCE.json`).

This dictionary is the `is_word` predicate that separates Regime A nonword
typos from Regime B real-word shifts. See `data/wordlists/README.md` for the
full rationale.

In [ ]:
import subprocess, sys

# Download the prebuilt SCOWL 2020.12.07 release from SourceForge (the
# built final/ word lists; NOT the en-wl/wordlist GitHub repo, which is
# SCOWL source + a Makefile with no prebuilt final/ directory).
run([
    "wget", "-q", "-O", "/tmp/scowl.tar.gz",
    "https://sourceforge.net/projects/wordlist/files/SCOWL/2020.12.07/scowl-2020.12.07.tar.gz/download",
])
run(["tar", "-xzf", "/tmp/scowl.tar.gz", "-C", "/tmp/"])

# Build the pinned vocabulary from the size-60 'words' lists only (english +
# american dialect) — deliberately narrower than SCOWL's own mk-list default,
# which also pulls in abbreviations/proper-names/upper/hacker-jargon/roman-
# numerals. See tools/build_dictionary.py and data/wordlists/README.md for
# why: those categories are not "real words" in the sense is_word needs.
run([
    PYTHON_BIN, "-u", "tools/build_dictionary.py",
    "--scowl-path",     "/tmp/scowl-2020.12.07/final/",
    "--scowl-max-size", "60",
    "--scowl-dialect",  "american",
])

print("Dictionary ready — data/wordlists/en_us_pinned.txt")

### Cell 5: Generate rehearsal outputs, full roster

Mounts Google Drive (one auth popup) and runs every generation model in the
roster through `configs/rehearsal.yaml`, **one output directory per model**
(shard manifest ids are run_id-keyed, so models must not share a directory).
Outputs land in `MyDrive/glamor/results/rehearsal/<model>/`, which is what
makes the run survive VM recycling and portable to the USC cluster.

**Order:** ungated and small models first, so a partial T4 session still
yields analyzable output; the three fp16 7-8B models run last and are
*expected to fail on a T4* (recorded, loop continues). Run those on the
cluster against the same Drive-synced directories, or on an L4/A100.

**A model that fails** (gated access not granted, out of GPU memory) is
recorded and skipped; fix the cause and re-run this cell. Completed models
re-verify as no-ops via their on-disk rows; an interrupted model resumes
from its last flushed row.

**Parallelism is automatic on one GPU:** vLLM's continuous-batching scheduler
keeps the GPU saturated (every pending request is submitted up front;
requests are pre-sorted for prefix-cache locality; `max_model_len` is
measured from the actual request set instead of the model's spec sheet).
Request *construction* fans out across all CPU cores via loky. The only
manual parallelism is multi-GPU sharding on the cluster
(`--shard-index`/`--shard-count`).

In [ ]:
import pathlib
import subprocess

# Drive-backed output root: resume state is the output directory and nothing
# else, so putting it on Drive makes the run survive VM recycling and lets
# the USC cluster pick up the same directories.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = pathlib.Path("/content/drive/MyDrive/glamor/results/rehearsal")
except ImportError:
    OUTPUT_ROOT = pathlib.Path("results/rehearsal")
    print("WARNING: not on Colab, or Drive unavailable; outputs are local only")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Output root: {OUTPUT_ROOT}")

# Ungated/small first so a partial T4 session still yields analyzable output;
# the fp16 7-8B models last (they exceed a T4's 16 GB and belong to the
# cluster or an L4/A100 session).
REHEARSAL_ROSTER = [
    "qwen_1b5_pilot",   # ungated fp16 1.5B
    "llama_1b",         # gated   fp16 1B
    "llama_3b",         # gated   fp16 3B
    "qwen_7b_awq",      # ungated AWQ  7B
    "llama_8b_awq",     # ungated AWQ  8B
    "mistral_7b",       # gated   fp16 7B  (exceeds T4)
    "qwen_7b",          # ungated fp16 7B  (exceeds T4)
    "llama_8b",         # gated   fp16 8B  (exceeds T4)
]

git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip() or "unpinned"

completed, failed = [], []
for roster_key in REHEARSAL_ROSTER:
    print(f"\n{'=' * 70}\n  {roster_key}\n{'=' * 70}")
    try:
        run([
            PYTHON_BIN, "-u", "tools/run_generation.py",
            "--config",           "configs/rehearsal.yaml",
            "--model",            roster_key,
            "--output-directory", str(OUTPUT_ROOT / roster_key),
            "--dictionary",       "data/wordlists/en_us_pinned.txt",
            "--git-commit",       git_commit,
        ])
        completed.append(roster_key)
    except subprocess.CalledProcessError as error:
        failed.append(roster_key)
        print(f"[FAILED] {roster_key} (exit {error.returncode}); continuing")

print(f"\nCompleted ({len(completed)}/{len(REHEARSAL_ROSTER)}): {completed}")
if failed:
    print(f"Failed: {failed}")
    print("Fix the cause (gated access, GPU size) and re-run this cell, or run")
    print("the failed models on the cluster against the same directories;")
    print("completed models are skipped via their on-disk rows.")

### Cell 6: Combined analysis, report, download

**One** `run_analysis` call over every model's rehearsal generations: the
multi-model crossed GLMM and the quantization interaction only exist when all
models sit in a single analysis, so per-model analyses would defeat the
rehearsal's purpose. The glob also picks up sharded `_wXofY_` files written
by cluster workers. If any roster model is missing (e.g. the fp16 7-8B trio
on a T4-only session) the cell warns and analyzes what is there; re-run it
after the cluster fills in the rest.

**`build_report`** writes the self-contained tabbed dashboard
(`report.html`, works offline). Everything is zipped and downloaded as
`rehearsal_results.zip`; the raw generations also persist on Drive
regardless.

Analysis compute (BCa bootstrap via scipy, `glmer` via rpy2/lme4, 1000
quasi-Bayes mediation draws) is minutes of single-process CPU work; it is
not the bottleneck and needs no parallelism configuration.

In [ ]:
import pathlib
import zipfile

# One combined analysis over every model's rehearsal outputs. The glob
# matches both unsharded (rehearsal_generations.jsonl) and cluster-sharded
# (rehearsal_wXofY_generations.jsonl) files.
generation_paths = sorted(
    str(p) for p in OUTPUT_ROOT.glob("*/rehearsal*_generations.jsonl"))
present_models = sorted({pathlib.Path(p).parent.name for p in generation_paths})
print(f"Models in analysis ({len(present_models)}): {present_models}")
missing = sorted(set(REHEARSAL_ROSTER) - set(present_models))
if missing:
    print(f"WARNING: missing from analysis: {missing}")
    print("The combined GLMM and report cover only the models above;")
    print("re-run this cell once the cluster fills in the rest.")
if not generation_paths:
    raise RuntimeError("no rehearsal generations found; run Cell 5 first")

run([
    PYTHON_BIN, "-u", "tools/run_analysis.py",
    "--generations",      *generation_paths,
    "--output-directory", "analysis/rehearsal",
    "--config",           "configs/rehearsal.yaml",
])

run([
    PYTHON_BIN, "-u", "tools/build_report.py",
    "--generations",        *generation_paths,
    "--output",             str(OUTPUT_ROOT / "report.html"),
    "--config",             "configs/rehearsal.yaml",
    "--analysis-directory", "analysis/rehearsal",
])

zip_path = pathlib.Path("rehearsal_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in pathlib.Path("analysis/rehearsal").rglob("*"):
        if path.is_file():
            zf.write(path)
    zf.write(OUTPUT_ROOT / "report.html", "results/rehearsal/report.html")
    for pattern in ("*/rehearsal*.jsonl", "*/rehearsal*manifest.json"):
        for path in OUTPUT_ROOT.glob(pattern):
            zf.write(path, f"results/rehearsal/{path.parent.name}/{path.name}")

try:
    from google.colab import files
    files.download(str(zip_path))
    print("Downloaded rehearsal_results.zip")
except ImportError:
    print(f"Results at {zip_path.resolve()}")